# Agent Frameworks and Tool Use: Building Autonomous AI Systems

## Introduction: From Chatbots to Agents

Large language models are powerful, but on their own they have significant limitations. They cannot access current information, perform calculations reliably, interact with external systems, or take actions in the world. Agent frameworks transform static language models into dynamic systems that can use tools, make decisions, and accomplish complex multi-step tasks autonomously.

An AI agent is a system that perceives its environment, makes decisions, and takes actions to achieve goals. In the context of language models, this means a system that can reason about what needs to be done, decide which tools to use, execute those tools, and iterate based on results. This agent paradigm represents a fundamental shift from models that simply respond to prompts to systems that can plan and execute complex workflows.

### The Evolution of LLM Agents

Early language model applications were essentially sophisticated completion systems. You provided a prompt, the model generated text, and that was the end of the interaction. As models became more capable, researchers began exploring ways to extend their abilities through external tools and iterative reasoning.

The breakthrough came with frameworks like LangChain, AutoGPT, and BabyAGI that provided structured approaches to building agents. These frameworks handle the orchestration of tool calls, memory management, and iterative reasoning. They allow developers to focus on defining tools and goals rather than implementing the underlying agent machinery.

Function calling, introduced in models like GPT-3.5 and GPT-4, provided a standardized interface for tool use. Instead of trying to parse tool calls from unstructured text, models can now output structured JSON that specifies which function to call with what parameters. This makes agent systems more reliable and easier to build.

### What This Notebook Covers

Through this comprehensive guide, you will learn to build agent systems from first principles. We start by implementing a simple agent loop manually, understanding each component and how they fit together. You will see how agents perceive their environment, reason about actions, execute tools, and iterate based on feedback.

We then explore function calling and structured tool use. You will learn how to define tools that language models can invoke, how to specify function schemas, and how to handle function outputs. This forms the foundation for reliable agent systems.

Next, we examine different agent architectures. ReAct agents interleave reasoning and action. Plan-and-execute agents separate high-level planning from low-level execution. Multi-agent systems coordinate multiple specialized agents. Each architecture suits different types of tasks.

We also cover practical frameworks like LangChain, showing you how to leverage existing infrastructure rather than building everything from scratch. You will learn common patterns, best practices, and how to debug agent systems when they go wrong.

Finally, we address important considerations around reliability, safety, and control. Autonomous agents can take unexpected actions or get stuck in loops. Understanding failure modes and implementing appropriate guardrails is essential for production deployments.

By the end of this notebook, you will have both the conceptual understanding and practical skills to build sophisticated agent systems. You will know when agents are appropriate, how to implement them effectively, and how to avoid common pitfalls.

Let us begin by understanding the core components that make up an agent system.

In [ ]:
# Install required packages
# !pip install transformers torch openai langchain anthropic

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import List, Dict, Callable, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import json
import re
from abc import ABC, abstractmethod
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("\nCore Agent Concepts:")
print("  • Perception: Understanding the current state")
print("  • Reasoning: Deciding what to do next")
print("  • Action: Using tools to affect the environment")
print("  • Memory: Maintaining context across steps")
print("  • Iteration: Repeating until goal is achieved")
print("\nAgents transform language models from responders into actors")

## Part 1: Building an Agent from First Principles

Before using frameworks, let us build a simple agent from scratch to understand how all the pieces fit together. An agent has three core components: tools it can use, a reasoning system that decides which tools to use, and an execution loop that orchestrates everything.

The agent loop follows a simple pattern: observe the current state, reason about what to do, execute an action, observe the results, and repeat until the goal is achieved. This loop, sometimes called the perception-action cycle, is fundamental to all agent architectures.

In [ ]:
@dataclass
class Tool:
    """Represents a tool that an agent can use.
    
    Tools are functions that agents can call to interact with
    the environment or perform computations.
    """
    name: str
    description: str
    function: Callable
    parameters: Dict[str, str] = field(default_factory=dict)
    
    def execute(self, **kwargs) -> str:
        """Execute the tool with given parameters."""
        try:
            result = self.function(**kwargs)
            return str(result)
        except Exception as e:
            return f"Error executing {self.name}: {str(e)}"
    
    def get_schema(self) -> Dict:
        """Get tool schema for model consumption."""
        return {
            'name': self.name,
            'description': self.description,
            'parameters': self.parameters
        }


class AgentAction(Enum):
    """Types of actions an agent can take."""
    USE_TOOL = "use_tool"
    RESPOND = "respond"
    NEED_MORE_INFO = "need_more_info"


@dataclass
class AgentStep:
    """Represents a single step in agent execution."""
    thought: str
    action: AgentAction
    action_input: Optional[Dict] = None
    observation: Optional[str] = None


class SimpleAgent:
    """A simple agent built from first principles.
    
    This agent implements the basic perception-action loop:
    1. Perceive current state
    2. Reason about what to do
    3. Take action (use tool or respond)
    4. Observe results
    5. Repeat until goal achieved
    """
    
    def __init__(self, tools: List[Tool], model_name: str = 'gpt2', max_iterations: int = 5):
        """Initialize agent with available tools."""
        self.tools = {tool.name: tool for tool in tools}
        self.max_iterations = max_iterations
        self.history: List[AgentStep] = []
        
        # Initialize language model for reasoning
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
    
    def get_tools_description(self) -> str:
        """Generate description of available tools."""
        descriptions = []
        for tool in self.tools.values():
            params = ", ".join([f"{k}: {v}" for k, v in tool.parameters.items()])
            descriptions.append(f"  - {tool.name}({params}): {tool.description}")
        return "\n".join(descriptions)
    
    def create_agent_prompt(self, query: str, current_state: str = "") -> str:
        """Create prompt for agent reasoning.
        
        The prompt structure guides the agent through the reasoning process:
        1. Present the task
        2. Show available tools
        3. Show execution history
        4. Ask for next action
        """
        prompt = f"""You are an AI agent that can use tools to answer questions and solve problems.

Available Tools:
{self.get_tools_description()}

Task: {query}
"""
        
        if self.history:
            prompt += "\n\nExecution History:\n"
            for i, step in enumerate(self.history, 1):
                prompt += f"\nStep {i}:\n"
                prompt += f"  Thought: {step.thought}\n"
                prompt += f"  Action: {step.action.value}\n"
                if step.observation:
                    prompt += f"  Observation: {step.observation}\n"
        
        if current_state:
            prompt += f"\nCurrent State: {current_state}\n"
        
        prompt += "\n\nWhat should I do next? Think step by step.\nThought:"
        
        return prompt
    
    def generate_response(self, prompt: str, max_length: int = 150) -> str:
        """Generate response from language model."""
        inputs = self.tokenizer(prompt, return_tensors='pt', 
                               truncation=True, max_length=1024).to(device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=inputs['input_ids'].shape[1] + max_length,
                temperature=0.7,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return generated[len(prompt):].strip()
    
    def parse_action(self, response: str) -> Tuple[AgentAction, Optional[Dict]]:
        """Parse agent response to extract action and parameters.
        
        This is a simplified parser. In practice, you would use
        more sophisticated parsing or structured output.
        """
        # Look for tool usage patterns
        for tool_name in self.tools.keys():
            if tool_name.lower() in response.lower():
                # Try to extract parameters (simplified)
                return AgentAction.USE_TOOL, {'tool': tool_name}
        
        # Check if agent wants to respond
        if any(word in response.lower() for word in ['answer', 'response', 'result']):
            return AgentAction.RESPOND, None
        
        # Default to needing more info
        return AgentAction.NEED_MORE_INFO, None
    
    def execute_step(self, query: str, current_state: str = "") -> AgentStep:
        """Execute a single step of agent reasoning."""
        # Generate reasoning
        prompt = self.create_agent_prompt(query, current_state)
        thought = self.generate_response(prompt, max_length=100)
        
        # Parse action
        action, action_input = self.parse_action(thought)
        
        # Execute action if tool use
        observation = None
        if action == AgentAction.USE_TOOL and action_input:
            tool_name = action_input.get('tool')
            if tool_name in self.tools:
                # Execute tool (with simplified parameter extraction)
                observation = self.tools[tool_name].execute()
        
        step = AgentStep(
            thought=thought,
            action=action,
            action_input=action_input,
            observation=observation
        )
        
        return step
    
    def run(self, query: str) -> str:
        """Run the agent loop until completion or max iterations."""
        print(f"\nStarting agent execution for query: {query}")
        print("=" * 80)
        
        self.history = []
        current_state = ""
        
        for i in range(self.max_iterations):
            print(f"\nIteration {i + 1}:")
            print("-" * 80)
            
            # Execute step
            step = self.execute_step(query, current_state)
            self.history.append(step)
            
            # Display step
            print(f"Thought: {step.thought[:200]}...")
            print(f"Action: {step.action.value}")
            if step.observation:
                print(f"Observation: {step.observation}")
            
            # Update state
            if step.observation:
                current_state = step.observation
            
            # Check if done
            if step.action == AgentAction.RESPOND:
                print("\nAgent completed task!")
                return step.thought
        
        print("\nMax iterations reached.")
        return "Could not complete task within iteration limit."


# Define some simple tools
def calculator(expression: str) -> float:
    """Evaluate a mathematical expression."""
    try:
        # Safe evaluation of basic math
        return eval(expression, {"__builtins__": {}}, {})
    except:
        return "Error: Invalid expression"

def search(query: str) -> str:
    """Simulate searching for information."""
    # In practice, this would call an actual search API
    return f"Search results for '{query}': [Example results would appear here]"

# Create tools
tools = [
    Tool(
        name="calculator",
        description="Evaluates mathematical expressions",
        function=calculator,
        parameters={"expression": "string (math expression)"}
    ),
    Tool(
        name="search",
        description="Searches for information on the internet",
        function=search,
        parameters={"query": "string (search query)"}
    )
]

# Create and run agent
agent = SimpleAgent(tools, max_iterations=3)

print("\n" + "="*80)
print("SIMPLE AGENT DEMONSTRATION")
print("="*80)

query = "What is 15 multiplied by 23?"
result = agent.run(query)

print("\n" + "="*80)
print("FINAL RESULT:")
print(result)

print("\n" + "="*80)
print("KEY INSIGHTS ABOUT AGENTS")
print("="*80)
print("""
Core Agent Components:
  • Tools: Functions the agent can execute
  • Reasoning: LLM decides which tool to use
  • Execution: Tool is called with parameters
  • Observation: Results feed back into reasoning
  • Memory: History maintained across iterations

Agent Loop Pattern:
  1. Perceive: Understand current state and goal
  2. Reason: Decide what action to take
  3. Act: Execute chosen tool or respond
  4. Observe: See results of action
  5. Repeat: Continue until goal achieved

Challenges:
  • Parsing: Extracting structured actions from text
  • Reliability: Ensuring correct tool usage
  • Termination: Knowing when to stop iterating
  • Error handling: Recovering from tool failures
  • Context length: Managing long execution histories

Why Agents Are Powerful:
  • Extend LLM capabilities beyond text generation
  • Enable interaction with external systems
  • Support multi-step reasoning and planning
  • Can adapt behavior based on observations
  • Bridge the gap between language and action
""")

## Conclusion

Throughout this notebook, we have explored the fundamental concepts and practical implementations of AI agent systems. We have built agents from first principles, understanding each component of the perception-action loop. We have implemented function calling to enable reliable tool use. We have explored different agent architectures and frameworks that make building agents more practical.

The key insights are:

**Agents extend capabilities.** By giving language models access to tools and external systems, we transform them from static text generators into dynamic problem solvers that can interact with the world.

**Architecture matters.** Different agent architectures suit different tasks. ReAct works well for exploratory tasks. Plan-and-execute excels at complex multi-step problems. Choose architecture based on your use case.

**Reliability is challenging.** Agents can fail in unexpected ways. Robust error handling, validation, and guardrails are essential for production systems.

**Frameworks help but add complexity.** Tools like LangChain provide valuable infrastructure but also introduce dependencies and abstractions. Understand what they do under the hood.

**Control and safety matter.** Autonomous agents can take unexpected actions. Implementing appropriate controls and monitoring is not optional.

As you build agent systems, remember that we are still in the early days of this technology. Patterns and best practices continue to evolve. Stay curious, experiment with new approaches, and always think critically about when agents add value versus when simpler approaches suffice. The goal is not to use agents everywhere, but to use them effectively where they provide genuine benefits.
""")